<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B08%5D%20-%20Ingenieria_de_Variables_II/%5B01%5D%20-%20Notebooks/E1_PCA_en_10_lineas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E1 · PCA en 10 líneas - Ingeniería de Variables II

## Introducción

> *Más columnas no significa mejor modelo.*

Cada variable es una **dimensión**. Si tienes 300 columnas, cada fila vive en un espacio de
300 dimensiones, y eso trae problemas (ruido, sobreajuste, coste). La **reducción de
dimensionalidad** busca pasar de muchas columnas a unas pocas que conservan **casi toda la
información útil**.

Es como una foto: aplasta un objeto 3D en 2D, pierdes una dimensión pero sigues reconociendo
a la persona. PCA hace algo parecido: proyecta a menos variables conservando la **varianza
que importa**.

En este primer ejercicio, en muy pocas líneas, vamos a:

1. Generar un dataset **ancho** (muchas columnas correlacionadas).
2. **Escalar** (paso clave antes de PCA).
3. Aplicar **PCA** y mirar la **varianza explicada**.
4. Dibujar el **scree plot** y la varianza acumulada.
5. Decidir **con cuántas componentes** nos quedamos.

## Objetivos del ejercicio

- Entender qué es una **componente principal** y la **varianza explicada**.
- Ver **por qué hay que escalar** antes de PCA.
- Leer un **scree plot** y la **varianza acumulada** para elegir el número de componentes.
- Comprobar que **pocas componentes** conservan casi toda la información.

## Descripción del dataset (sensores, dataset "ancho")

Para esta sesión usamos un dataset **sintético y reproducible** que imita un caso muy
común: **muchísimas columnas pero pocas dimensiones reales**. Piensa en cientos de
sensores que, en el fondo, miden unas pocas cosas (temperatura, presión, vibración...).

Lo generamos dentro del propio notebook con `generar_datos_anchos`, así que es
autocontenido en Colab. Por dentro:

- Hay unos pocos **factores latentes** (la "verdad" oculta) que generan la señal.
- Cada **columna `sensor_XXX`** es una mezcla de esos factores más algo de ruido, así que
  muchas columnas **dicen casi lo mismo** (están muy correlacionadas).
- Las columnas tienen **escalas muy distintas** a propósito (de 1 a 1000), para ver por
  qué hay que escalar antes de PCA.
- `target` es la etiqueta (la clase de cada fila).

> La idea de fondo de la clase: *más columnas no significa más información*. Aquí lo vemos
> en directo, porque la información real vive en muy pocas dimensiones.

### 1. Importar librerías necesarias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

plt.rcParams["figure.figsize"] = (10, 4)

### 2. Generar un dataset ancho

In [ ]:
import numpy as np
import pandas as pd

def generar_datos_anchos(n=800, n_cols=60, n_latentes=5, n_clases=2,
                         separacion=2.5, ruido=0.6, semilla=42):
    # Genera un dataset "ancho": muchas columnas (sensores) pero POCAS dimensiones
    # reales. Unos pocos factores latentes generan casi toda la informacion; el
    # resto de columnas son mezclas de esos factores mas ruido. Asi PCA puede
    # recuperar la estructura con pocas componentes.
    rng = np.random.default_rng(semilla)

    # Centros de cada clase en el espacio latente (grupos separados)
    centros = rng.normal(scale=separacion, size=(n_clases, n_latentes))
    y = rng.integers(0, n_clases, size=n)
    Z = centros[y] + rng.normal(size=(n, n_latentes))      # factor latente de cada fila

    # Cada columna observada = mezcla ponderada de los factores latentes + ruido
    cargas = rng.normal(size=(n_latentes, n_cols))
    X = Z @ cargas + ruido * rng.normal(size=(n, n_cols))

    # Escalas MUY distintas por columna (para motivar StandardScaler antes de PCA)
    escalas = rng.uniform(1, 1000, size=n_cols)
    X = X * escalas

    cols = [f"sensor_{i:03d}" for i in range(n_cols)]
    df = pd.DataFrame(X, columns=cols)
    df["target"] = y
    return df

In [ ]:
df = generar_datos_anchos(n=800, n_cols=60, n_latentes=5, semilla=42)
X = df.drop(columns="target")
y = df["target"]

print("Dimensiones del dataset:", X.shape, "(filas x columnas)")
print("La información vive en solo 5 dimensiones latentes.")
df.iloc[:5, list(range(4)) + [-1]]

### 3. Paso clave: escalar ANTES de PCA

PCA se fija en la **varianza**. Si una columna está en miles (por su escala) y otra en
decenas, la grande se "come" a las demás solo por su tamaño, no porque tenga más señal.
Mira lo dispares que son las escalas de nuestras columnas:

In [ ]:
print("Desviación típica por columna (resumen):")
print(X.std().describe()[["min", "50%", "max"]].round(2))
print("\nUnas columnas varían muchísimo más que otras solo por su escala.")

# Escalamos: media 0 y desviación 1 en cada columna
X_esc = StandardScaler().fit_transform(X)
print("\nTras escalar, todas las columnas tienen desviación ~1.")
print("Desviación media tras escalar:", round(X_esc.std(axis=0).mean(), 3))

### 4. PCA en pocas líneas

In [ ]:
pca = PCA()                 # sin fijar n_components: las calcula todas
pca.fit(X_esc)

evr = pca.explained_variance_ratio_      # varianza explicada por cada componente
acum = np.cumsum(evr)

print("Varianza explicada por las primeras 8 componentes:")
print(np.round(evr[:8], 3))
print("\nLa primera componente sola explica el", f"{evr[0]*100:.1f}%", "de la varianza.")

### 5. Varianza explicada: scree plot y acumulada

In [ ]:
k = 15  # mostramos solo las primeras componentes
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

ax[0].bar(range(1, k + 1), evr[:k], color="#2980b9")
ax[0].set_title("Scree plot: varianza por componente")
ax[0].set_xlabel("Componente")
ax[0].set_ylabel("Proporción de varianza")

ax[1].plot(range(1, len(acum) + 1), acum, marker="o", color="#27ae60")
ax[1].axhline(0.90, ls="--", color="gray", label="90%")
ax[1].axhline(0.95, ls="--", color="orange", label="95%")
ax[1].set_title("Varianza explicada acumulada")
ax[1].set_xlabel("Nº de componentes")
ax[1].set_ylabel("Varianza acumulada")
ax[1].legend()

plt.tight_layout()
plt.show()

Fíjate en el **codo** del scree plot: a partir de unas pocas componentes, cada nueva apenas
aporta. Eso indica que la información real está en pocas dimensiones (las latentes).

### 6. ¿Cuántas componentes nos quedamos?

In [ ]:
n90 = int(np.argmax(acum >= 0.90)) + 1
n95 = int(np.argmax(acum >= 0.95)) + 1

print(f"Dimensiones originales: {X.shape[1]} columnas")
print(f"Para conservar el 90% de la varianza bastan {n90} componentes")
print(f"Para conservar el 95% de la varianza bastan {n95} componentes")
print(f"\nHemos pasado de {X.shape[1]} columnas a {n90}: una reducción enorme.")

### 7. Comprobación: pocas componentes, casi toda la información

In [ ]:
pca_reducido = PCA(n_components=n90)
X_red = pca_reducido.fit_transform(X_esc)

print("Forma tras reducir:", X_red.shape)
print("Varianza conservada:", f"{pca_reducido.explained_variance_ratio_.sum()*100:.1f}%")
print("Cada fila pasa de", X.shape[1], "números a solo", X_red.shape[1], "sin perder casi nada.")

### 8. ¿Y si NO escalamos? (el error típico)

Si aplicamos PCA sin escalar, la columna de mayor escala domina y la "varianza explicada"
engaña: la primera componente parece explicarlo casi todo, pero solo está siguiendo a la
columna más grande, no a la estructura real.

In [ ]:
pca_sin_escalar = PCA().fit(X.values)     # OJO: sobre los datos SIN escalar
evr_sin = pca_sin_escalar.explained_variance_ratio_

print(f"SIN escalar -> 1ª componente explica el {evr_sin[0]*100:.1f}% (engañoso: manda la escala)")
print(f"CON escalar -> 1ª componente explica el {evr[0]*100:.1f}% (refleja la estructura real)")
print("\nMoraleja: escala SIEMPRE antes de PCA.")

### Reflexión

1. ¿Qué representa la **primera componente principal** en este dataset?
2. ¿Por qué el **scree plot** tiene un codo tan claro alrededor de 5 componentes?
3. ¿Qué pasaría con la varianza explicada si subimos el `ruido` del generador?
4. ¿Por qué escalar antes de PCA cambia tanto el resultado?
5. Si el siguiente paso fuera **visualizar** los datos, ¿cuántas componentes elegirías?